# Metaflora Incubus v1 — recover GGUF
One cell restores the trained adapter, exports Q5_K_M, runs the committed benchmark and syncs the private artifact. Training is not repeated.

In [ ]:
import os
import shutil
import site
import subprocess
import sys
from pathlib import Path

trusted_code_revision = "57d84e452cf153ed85c4652d80763f480b94bee7"
kaggle_root = Path("/kaggle/working")
is_kaggle = kaggle_root.is_dir()
working_root = kaggle_root if is_kaggle else Path("/content")
repository = str(working_root / "metaflora-incubus")
os.chdir(working_root)
shutil.rmtree(repository, ignore_errors=True)
subprocess.run(["git", "init", repository], check=True)
subprocess.run(["git", "-C", repository, "remote", "add", "origin", "https://github.com/metaflora-app/metaflora-incubus.git"], check=True)
subprocess.run(["git", "-C", repository, "fetch", "--depth", "1", "origin", trusted_code_revision], check=True)
subprocess.run(["git", "-C", repository, "checkout", "--detach", "FETCH_HEAD"], check=True)
checked_out = subprocess.run(["git", "-C", repository, "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
if checked_out != trusted_code_revision:
    raise RuntimeError("trusted code revision verification failed")
subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", f"{repository}/requirements/recovery-linux.lock"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", repository], check=True)
library_roots = sorted({str(path) for root in site.getsitepackages() for path in Path(root).glob("nvidia/*/lib") if path.is_dir()})
runtime_environment = dict(os.environ)
gpu_query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], check=True, capture_output=True, text=True)
gpu_rows = [row.strip() for row in gpu_query.stdout.splitlines() if row.strip()]
if is_kaggle and (len(gpu_rows) != 2 or any("T4" not in row for row in gpu_rows)):
    raise RuntimeError("Kaggle recovery requires the GPU T4 x2 accelerator")
if is_kaggle:
    runtime_environment["CUDA_VISIBLE_DEVICES"] = "0,1"
runtime_environment["LD_LIBRARY_PATH"] = os.pathsep.join([*library_roots, os.environ.get("LD_LIBRARY_PATH", "")])
source_root = f"{repository}/src"
if source_root not in sys.path:
    sys.path.insert(0, source_root)
os.chdir(repository)
from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap
if is_kaggle:
    from kaggle_secrets import UserSecretsClient
    encoded_bootstrap = UserSecretsClient().get_secret("INCUBUS_BOOTSTRAP")
else:
    from google.colab import userdata
    encoded_bootstrap = userdata.get("INCUBUS_BOOTSTRAP")
if not isinstance(encoded_bootstrap, str) or not encoded_bootstrap.strip():
    raise RuntimeError("INCUBUS_BOOTSTRAP is empty")
bootstrap_payload = decrypt_cloud_bootstrap(Path("configs/cloud/bootstrap-v1.enc").read_bytes(), encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
runtime_environment.update(os.environ)
if is_kaggle and shutil.disk_usage("/tmp").free < 64 * 1024**3:
    raise RuntimeError("Kaggle temporary disk needs at least 64 GiB free")
workspace_root = "/tmp/incubus-work" if is_kaggle else "/content/incubus-work"
subprocess.run([sys.executable, "scripts/recover_free_gpu.py", "--run-id", "incubus-v1-run", "--parameter-count", os.environ["INCUBUS_PARAMETER_COUNT"], "--checkpoint-location", "metaflora/incubus-checkpoints", "--checkpoint-branch", "incubus-training-v1", "--workspace-root", workspace_root], check=True, env=runtime_environment)
